<img src="img/swdb_logo.jpg" width="900" style="display: block; margin: 0 auto;">

<h1 align="center">Connectomics Exercise 2: How robust is the structure&ndash;function result?</h1>
<h3 align="center">Summer Workshop on the Dynamic Brain 2026</h3>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<p>Module 2b fixed three choices to keep the logic simple: one stimulus condition
(<code>natural_images</code>), one similarity measure (Pearson correlation of the full
&Delta;F/F trace), and one cell type (L3-IT). It then reported a result: connected pairs
are more correlated than testable unconnected pairs, by more than a distance-matched null
model can explain.
<p>A result that holds only for choices you make may not be a result about the brain, it may
be a result about your choices. This exercise relaxes them one at a time.
<ol>
<li><b>Does the choice of stimulus condition matter?</b>
<li><b>Which condition drives the strongest correlations</b> &mdash; and is that the same as
the one where correlation best predicts connectivity?
<li><b>Does the null-model result survive in every condition?</b>
<li><b>Which cell types show the effect at all?</b>
</ol>
<p>Look at each figure before reading the number below it. Every task here has a picture that
carries the answer; the tests are there to say whether what you saw is more than sampling
noise.
<p>The preliminaries rebuild Module 2b Parts 1&ndash;3 &mdash; nothing new, so run them and
move on.

</div>

In [ ]:
import sys
from os.path import join as pjoin

mat_version = 1196

# Identifiers within the Common Connectivity dataset
project_id = "v1dd"
synapse_dataset_id = f"v1dd_{mat_version}_em"
synapse_feature_matrix_id = f"v1dd_{mat_version}_synapse_features"
axon_dataset_id = f"v1dd_{mat_version}_proofread_axons"
dendrite_dataset_id = f"v1dd_{mat_version}_proofread_dendrites"

sys.path.append(pjoin("..", "utils"))

import itertools

import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
import tqdm
from connects_common_connectivity.io import DatasetReader, read_synapse_table
from matplotlib import pyplot as plt
from scipy import spatial, stats

from paths import resolve_data_root, resolve_dataset_dir
from utils import filter_synapse_table

data_root = resolve_data_root(f"v1dd_{mat_version}_ccm")

# The EM side: cells, cell types and synapses, in Common Connectivity format
ccm_dir = resolve_dataset_dir(f"v1dd_{mat_version}_ccm", root=data_root)

# The two-photon side: its own dataset on CodeOcean, but it sits next to the older feather
# tables if you downloaded those together. Accept either.
functional_dir = resolve_dataset_dir(
    f"v1dd_{mat_version}_coreg_functional_correlation",
    f"v1dd_{mat_version}",
    root=data_root,
)

print(f"ccm_dir        {ccm_dir}")
print(f"functional_dir {functional_dir}")

In [ ]:
# The EM side: which cells are proofread, where their somas are, what type they are, and
# every synapse between them.
reader = DatasetReader(ccm_dir)

proofread_axons = reader.read_dataset(axon_dataset_id)
proofread_dendrites = reader.read_dataset(dendrite_dataset_id)

axon_proof_root_ids = proofread_axons["dataitem_id"].cast(pl.UInt64).to_numpy()
dendrite_proof_root_ids = proofread_dendrites["dataitem_id"].cast(pl.UInt64).to_numpy()

cell_df = proofread_dendrites.select(
    pl.col("dataitem_id").cast(pl.UInt64).alias("pt_root_id"),
    # transformed coordinates, already in µm
    pl.col("soma_transformed_x").alias("pt_position_trform_x"),
    pl.col("soma_transformed_y").alias("pt_position_trform_y"),
    pl.col("soma_transformed_z").alias("pt_position_trform_z"),
    pl.col("soma_volume").alias("volume"),
    pl.col("v1dd_cell_types_level_1").alias("cell_type_coarse"),
    pl.col("v1dd_cell_types_level_2").alias("cell_type"),
).to_pandas()

synapse_data = read_synapse_table(
    project_id,
    dataset_id=synapse_dataset_id,
    features=True,
    feature_matrix_id=synapse_feature_matrix_id,
    output_root=ccm_dir,
)

syn_df = (
    synapse_data.with_columns(
        pl.col("id").cast(pl.UInt64),
        pl.col("presynaptic_cell").cast(pl.UInt64),
        pl.col("postsynaptic_cell").cast(pl.UInt64),
    )
    .rename(
        {
            "presynaptic_cell": "pre_pt_root_id",
            "postsynaptic_cell": "post_pt_root_id",
        }
    )
    .select(["id", "pre_pt_root_id", "post_pt_root_id", "size"])
    .to_pandas()
)

print(f"{len(axon_proof_root_ids):>10,} cells with proofread axons")
print(f"{len(dendrite_proof_root_ids):>10,} cells with acceptable dendrites")
print(f"{len(syn_df):>10,} synapses")

In [ ]:
# The two-photon side: one row per coregistered pair, one column per stimulus condition,
# each entry the correlation of the two ΔF/F traces over the frames of that condition.
# Built in ../supplement/Functional Data Cell-Cell Correlations.ipynb
corr_coreg_df = pd.read_feather(
    f"{functional_dir}/cell_cell_correlations_by_stimulus_coregistered.feather"
)

# Everything that is not an identifier column is a stimulus condition.
# `errors="ignore"` because which metadata columns are present depends on which version
# of the table you have.
stimulus_conditions = corr_coreg_df.columns.drop(
    [
        "pre_pt_root_id",
        "post_pt_root_id",
        "column",
        "volume",
        "pre_plane",
        "pre_roi",
        "post_plane",
        "post_roi",
    ],
    errors="ignore",
)

print(f"{len(corr_coreg_df):,} coregistered pairs")
list(stimulus_conditions)

In [ ]:
# Two helpers from Module 2b, unchanged.


def calculate_lateral_distances(pre_cell_df, post_cell_df=None):
    """Lateral distances in µm between all pairs of neurons."""
    if post_cell_df is None:
        post_cell_df = pre_cell_df

    pos_cols = ["pt_position_trform_x", "pt_position_trform_z"]
    lateral_distances = spatial.distance.cdist(
        np.array(pre_cell_df[pos_cols]), np.array(post_cell_df[pos_cols])
    )

    id_pairs = list(
        itertools.product(pre_cell_df["pt_root_id"], post_cell_df["pt_root_id"])
    )
    lateral_distance_df = pd.DataFrame(
        id_pairs, columns=["pre_pt_root_id", "post_pt_root_id"]
    )
    lateral_distance_df["distance"] = lateral_distances.flatten()

    # drop self-pairs
    return lateral_distance_df[
        lateral_distance_df["pre_pt_root_id"] != lateral_distance_df["post_pt_root_id"]
    ]


def compare_distributions(sample_a, sample_b, name_a="a", name_b="b", verbose=True):
    """Mann-Whitney U test between two samples, with its effect size.

    Returns (p, auc), where auc is the probability that a random draw from
    `sample_a` exceeds a random draw from `sample_b`. 0.5 means no difference.
    """
    sample_a = np.asarray(sample_a)
    sample_b = np.asarray(sample_b)
    u, p = stats.mannwhitneyu(sample_a, sample_b, alternative="two-sided")
    auc = u / (len(sample_a) * len(sample_b))

    if verbose:
        print(
            f"{name_a} (n={len(sample_a):,}) vs {name_b} (n={len(sample_b):,}): "
            f"AUC {auc:.3f}, p {p:.2g}"
        )
    return p, auc

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3> Module 2b Parts 1&ndash;3 in one function </h3>
<p>Convenience function to repeat the analysis across cell types.
<p><code>build_pair_table</code> returns one row per <i>testable</i> pair &mdash; presynaptic
cell has a proofread axon, postsynaptic cell has an acceptable dendrite, both are
coregistered &mdash; carrying:
<ul>
<li>every stimulus condition's activity correlation,
<li><code>distance</code>, the lateral separation of the two somas,
<li><code>connected</code>, whether a synapse was found,
<li><code>connection_probability</code>, what the measured distance-dependence of
connectivity predicts for a pair this far apart. This is the column the null model samples
with.
</ul>
<p>Tasks 3 and 4 both depend on knowing what is in the table. The numbers printed below it
should match Module 2b Part 3 &mdash; if they do not, something went wrong and everything
after it is suspect.

</div>

In [ ]:
def build_pair_table(pre_cell_df, post_cell_df=None, max_distance=500, n_bins=100):
    """Module 2b Parts 1-3, as one call. See the text above for the columns.

    Returns (full_df, conn_df, testable_distance_df).
    """
    if post_cell_df is None:
        post_cell_df = pre_cell_df

    # -- Part 1: distances, restricted to pairs whose connectivity is testable
    distance_df = calculate_lateral_distances(pre_cell_df, post_cell_df)

    pre_root_ids = pre_cell_df["pt_root_id"][
        np.isin(pre_cell_df["pt_root_id"], axon_proof_root_ids)
    ]
    post_root_ids = post_cell_df["pt_root_id"][
        np.isin(post_cell_df["pt_root_id"], dendrite_proof_root_ids)
    ]
    testable_distance_df = distance_df[
        np.isin(distance_df["pre_pt_root_id"], pre_root_ids)
        & np.isin(distance_df["post_pt_root_id"], post_root_ids)
    ]

    # -- Part 1: connections, as summed synapse size per pair
    conn_df = (
        filter_synapse_table(syn_df, pre_root_ids, post_root_ids)
        .groupby(["pre_pt_root_id", "post_pt_root_id"])["size"]
        .sum()
        .reset_index()
    )
    conn_dist_df = pd.merge(
        conn_df, testable_distance_df, on=["pre_pt_root_id", "post_pt_root_id"]
    )

    # -- Part 2: the functional side, restricted the same way
    corr_testable_df = corr_coreg_df[
        np.isin(corr_coreg_df["pre_pt_root_id"], pre_root_ids)
        & np.isin(corr_coreg_df["post_pt_root_id"], post_root_ids)
    ]

    full_df = pd.merge(
        corr_testable_df, testable_distance_df, on=["pre_pt_root_id", "post_pt_root_id"]
    )
    full_df = pd.merge(
        full_df, conn_df, on=["pre_pt_root_id", "post_pt_root_id"], how="left"
    ).fillna({"size": 0})
    full_df["connected"] = full_df["size"] > 0

    # -- Part 3: connection probability per distance bin, from the measurement itself
    distance_bins = np.linspace(0, max_distance, n_bins + 1)
    probability_df = pd.DataFrame(
        {
            "bin_id": np.arange(n_bins),
            "connection_probability": (
                np.histogram(conn_dist_df["distance"], distance_bins)[0]
                / np.histogram(testable_distance_df["distance"], distance_bins)[0]
            ),
        }
    )
    full_df["bin_id"] = np.digitize(full_df["distance"], distance_bins) - 1
    full_df = pd.merge(full_df, probability_df, on="bin_id")

    return full_df, conn_df, testable_distance_df


# Rebuilding the Module 2b analysis set: L3-IT to L3-IT
sub_cell_df = cell_df[cell_df["cell_type"] == "L3-IT"]

full_df, conn_df, testable_distance_df = build_pair_table(sub_cell_df)

print(f"{len(sub_cell_df):>10,} L3-IT cells")
print(f"{len(testable_distance_df):>10,} testable L3-IT pairs")
print(f"{len(full_df):>10,} of those are coregistered")
print(f"{int(full_df['connected'].sum()):>10,} of those are connected")

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<h3> Task 1: Does the choice of stimulus condition matter? </h3>
<p>The seven conditions are seven different measurements. Whether the
statistics computed from them agree is open. If correlations under
drifting gratings correlate with correlations under natural movies, Module 2b's choice 
of <code>natural_images</code> and results would generalise; if not, the result depends on
which condition was picked.
<p>Plot the correlation <i>between the stimulus conditions</i>, across pairs:
<code>corr_coreg_df[stimulus_conditions].corr()</code> as a heatmap
(<code>sns.heatmap</code>, <code>vmin=0</code>, <code>vmax=0.5</code>,
<code>annot=True</code>). Each <i>row</i> of the dataframe is a pair of cells and each
<i>column</i> a condition, so this is a 7&times;7 matrix over conditions, not over cells.
<p>What do you expect before you run it? Which two conditions should agree most?

</div>

In [ ]:
# Your code here

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<p>Two things to read off the matrix. First the overall level: values well below 1 mean the
conditions are <i>not</i> interchangeable &mdash; knowing a pair's correlation under one
condition leaves most of the variance under another unexplained. Second the structure:
<code>drifting_gratings_full</code> and <code>drifting_gratings_windowed</code> are versions
of the same stimulus, so their correlations should agree better with each other than with
other conditions. That is a useful sanity check.
<p>The practical consequence: seven conditions are seven tests, so a result that is strong in
one condition and absent in another is a finding to be explained.

</div>

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<p><b>Discussion</b> The conditions differ in what they show and for how long. From one
session, in frames (the functional data notebook prints these):
<pre><code>natural_movie                5992
drifting_gratings_windowed   3479
drifting_gratings_full       3477
locally_sparse_noise         3270
spontaneous                  1820
natural_images               1810
natural_images_12             922
</code></pre>
<p>They also differ in how much repetition they contain. <code>natural_images</code> shows
118 images eight times each in randomised order; <code>natural_images_12</code> shows only
12 images in a frozen sequence repeated 40 times. Same image set, same 3 Hz presentation
rate, very different experiments.
<p>A correlation estimated from 922 frames is noisier than one from 5,992, and noise in
either of two vectors pulls the correlation between them toward zero. Two separate things
you could measure:
<ol>
<li><b>Is the disagreement just length?</b> Subsample every condition down to the shortest,
recompute, and see what survives.
<li><b>What is the most any condition could agree with anything?</b> Split one condition's
frames in half, compute the pair-correlation vector from each half, and correlate the halves.
That split-half reliability is an <i>upper bound</i> on any correlation involving that
condition &mdash; it is not an answer about length, it is the ceiling.
</ol>
<p>Both need the &Delta;F/F traces, so both start from the functional data notebook, which we may tap into another time.

</div>

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<h3> Task 2: Which condition drives the strongest correlations? </h3>
<p>Two different things could be meant by "the strongest condition", and keeping them apart
is the point of this task: the <b>overall correlation level</b> during that stimulus,
connected or not; and <b>how well correlation separates connected from unconnected pairs</b>,
the Module 2b result. A stimulus can raise both distributions equally while separating them
no better at all.
<p><b>Task 2a</b> Overlay the correlation distributions for all seven conditions on one
axis, as in Module 2b Figure 5. Use <code>corr_coreg_df</code> (all coregistered pairs),
<code>sns.histplot</code> with <code>stat="probability"</code>,
<code>element="step"</code>, <code>fill=False</code> and shared <code>bins</code>.

</div>

In [ ]:
# Your code here

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<p><b>Task 2b</b> Now the other sense of "strongest". For each condition, plot the mean
correlation of the connected pairs in <code>full_df</code> against the mean of the
testable-but-unconnected pairs, as two dots joined by a line &mdash; the length of the line
is the effect. Sort the conditions by that gap.
<p>Then print the test beside it: <code>compare_distributions(...)</code> returns
<code>(p, auc)</code>, and the AUC is the readable one &mdash; the probability that a random
connected pair is more correlated than a random unconnected one, where 0.5 is no separation.
<p>Is the order the same as the order of overall correlation level in Task 2a?

</div>

In [ ]:
# Your code here

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<p>Compare <code>mean_all</code> (how correlated the population is) against
<code>auc</code> (how well correlation separates connected pairs). If the two orderings
differ, the two senses of "strongest" are different quantities, and any claim of the form
"the effect is strongest during X" has to say which one it means.
<p>Watch the sample sizes: only a few hundred pairs are both coregistered and connected, so
these AUCs carry real uncertainty. An AUC of 0.58 against 0.55 is not a ranking worth
defending without a confidence interval &mdash; bootstrap the pairs if you want one.

</div>

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<h3> Task 3: Does the result survive the distance-matched null? </h3>
<p>Task 2b's gap has the weakness Module 2b Part 3 was written to address: connected pairs
are also <i>nearby</i> pairs, and nearby cells are more correlated for reasons that have
nothing to do with being wired together. Every per-condition gap inherits that confound. So
repeat Part 3 per condition: the null holds the distance dependence fixed &mdash; same number
of connections, each pair drawn with the probability its distance implies &mdash; and asks
whether the observed mean correlation of connected pairs is still surprising.
<p><b>Task 3a</b> Write the sampling loop as a function. For <code>n_samples</code>
iterations and a given <code>condition</code>: draw <code>full_df["connected"].sum()</code>
pairs from <code>full_df.index</code> with <code>np.random.choice</code>, using
<code>connection_probability</code> normalised to sum to 1 as <code>p</code>, and record the
mean of <code>condition</code> over the drawn pairs. Return the array of means.
<p>This is Module 2b's Part 3 loop with the stimulus made an argument.

</div>

In [ ]:
def sample_null_means(full_df, condition, n_samples=2_000, seed=0):
    """Mean correlation of connected pairs under `n_samples` distance-matched connectomes."""
    # Your code here

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<p><b>Task 3b</b> Run it for every condition and plot each null distribution against its
observed value. Centre every null on its own mean, so the seven are comparable: plot the
2,000 sampled means minus the null mean, and the observed mean minus the same null mean as a
single point. A condition shows the effect when its point sits clear of its distribution.
<p>Then report the two numbers per condition: the Z-score
<p align="center"><i>Z</i> = (observed &minus; null mean) / null sd</p>
<p>and the empirical <i>p</i>-value
<p align="center"><i>p&#770;</i> = (1 + #{<i>b</i> : <i>T<sub>b</sub></i> &ge;
<i>t</i><sub>obs</sub>}) / (1 + <i>B</i>)</p>
<p>for <i>B</i> samples. (2,000 samples per condition is enough to rank them; Module 2b used
10,000 for its single condition.)

</div>

In [ ]:
# Your code here

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<p>Read the picture first: how far each red point sits outside its grey distribution is the
result, and the width of the grey is why <i>Z</i> is not simply the size of the effect. A
condition with tightly clustered nulls gets a large <i>Z</i> from a small offset, and the
width depends on how spread that condition's correlations are and on how many connected
pairs there are &mdash; not on biology. So compare the offsets as well as the Z-scores.

</div>

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<p><b>Discussion</b> <code>spontaneous</code> is a grey screen: no visual stimulus, just
whatever the cortex does on its own. Suppose it scores as high as the visual conditions. Is
that evidence that the effect has nothing to do with visual responses, evidence that
spontaneous activity reflects the same wiring, or evidence of a problem with the analysis?
What further measurement would separate those?

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3> Task 4: Which cell types show the effect? </h3>
<p>Everything so far is L3-IT wired to L3-IT &mdash; one type, in one layer, connected to
itself. That is where the statistics are best and where this relationship has most often been
reported, which is exactly why it is a weak test of generality.
<p><code>build_pair_table</code> takes a presynaptic and a postsynaptic cell table, so
another combination is one line. What will bite is sample size, and the binding constraint is
<i>coregistration</i>: a pair is usable only if both cells were found in the two-photon volume
as well as the EM volume. The cell below counts what is available &mdash; and because the
two-photon volume was imaged in separate depth ranges, correlations exist between cells
recorded together and essentially not across them, so most cross-type combinations have no
pairs at all.

</div>

In [ ]:
coreg_ids = set(corr_coreg_df["pre_pt_root_id"]) | set(corr_coreg_df["post_pt_root_id"])

type_summary = (
    cell_df.assign(
        coregistered=cell_df["pt_root_id"].isin(coreg_ids),
        proofread_axon=cell_df["pt_root_id"].isin(axon_proof_root_ids),
    )
    .groupby(["cell_type_coarse", "cell_type"])
    .agg(
        cells=("pt_root_id", "size"),
        coregistered=("coregistered", "sum"),
        proofread_axon=("proofread_axon", "sum"),
    )
    .sort_values("coregistered", ascending=False)
)

type_of = cell_df.set_index("pt_root_id")["cell_type"]
pair_types = (
    pd.DataFrame(
        {
            "pre": corr_coreg_df["pre_pt_root_id"].map(type_of),
            "post": corr_coreg_df["post_pt_root_id"].map(type_of),
        }
    )
    .groupby(["pre", "post"], dropna=True)
    .size()
    .sort_values(ascending=False)
)

print("coregistered cells per type\n")
print(type_summary.head(8))
print("\ncoregistered pairs per type combination\n")
print(pair_types.head(8))

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<p><b>Task 4a</b> Pick combinations that actually appear in the pair-type table and rerun
the analysis on each: Task 2b's connected-vs-unconnected comparison and Task 3b's null
model, for <code>natural_images</code>. Include L3-IT &rarr; L3-IT so there is a reference
point, and report the number of connected pairs alongside every result &mdash; a Z-score
computed from twelve pairs is not a finding.

</div>

In [ ]:
# Your code here

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<p><b>Task 4b</b> One figure for the whole exercise: which cell-type combinations show the
relationship and which do not. Plot the AUC from Task 2b and the Z-score from Task 3b side
by side, one row per combination, with the number of connected pairs in the label. Mark the
no-effect line in each panel (AUC 0.5, Z 0).

</div>

In [ ]:
# Your code here

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<p>Both panels are coloured by the same rule &mdash; whether the combination cleared the null
model &mdash; which makes the left panel's job visible: an AUC above 0.5 is not by itself
evidence, because distance alone can produce it.
<p>Whatever you found, the honest write-up names the sample size in the same sentence as the
effect. "No effect" and "not enough pairs to detect the effect Module 2b found" are different
claims, and telling them apart needs a power calculation, not a <i>p</i>-value: given the AUC
measured for L3-IT, how many connected pairs would you need to detect it at <i>p</i> &lt;
0.05? If your new combination has fewer than that, you have learned nothing about it yet.

</div>

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<p><b>Bonus</b> Two more assumptions, each a small change to <code>build_pair_table</code>:
<ul>
<li><b>Distance is lateral only.</b> The analysis uses tangential (<i>x</i>, <i>z</i>)
separation and ignores depth, on the grounds that cells of one type sit at roughly one depth.
That stops being true across layers. Add <code>pt_position_trform_y</code> for a full 3D
distance and rerun. Does the cross-type result change?
<li><b>Connections are binary.</b> <code>conn_df</code> carries summed synapse size, which
the analysis throws away with <code>size &gt; 0</code>. Are strongly connected pairs more
correlated than weakly connected ones? Both grow with proximity, so the distance confound is
back and needs the same treatment.
</ul>

</div>

## What to take away

1. **The stimulus conditions are not interchangeable** (Task 1), so the result is a statement
   about a stimulus, not about the cortex in general.
2. **"Strongest" is ambiguous** (Task 2). The condition with the highest correlations overall
   need not be the one where correlation best separates connected pairs, and those two
   readings support different claims.
3. **Ranking needs the null model** (Task 3) — and even then, part of a Z-score is the width
   of the null rather than the size of the effect, so read the offsets too.
4. **Generality is a sample-size question first** (Task 4). Outside L3-IT the counts fall
   fast, and "no effect" usually means "no power". The dataset also decides which questions
   can be asked: no inhibitory cell is coregistered, and cross-layer pairs mostly do not
   exist.

None of that weakens the Module 2b result. It locates it: connectivity and co-activity are
associated for L3-IT to L3-IT pairs, during natural images, by more than a distance-matched
null can produce. Every qualifier in that sentence was earned by one of the tasks above.